# Spiny dendrite morphology demo

Build a synthetic dendrite trunk with classical neck+head spines from explicit
parameters, write the subsystem files (`SWC` + `AZ` + `neckpoint`), visualize,
and optionally attach a sink with the existing geometry API.

In [1]:
from swctools import SWCModel, PointSet, plot_model

from toric_spines_sim.geometry.dendrite import (
    SpinyDendriteParams,
    build_spiny_dendrite,
    write_subsystem,
)
from toric_spines_sim.geometry.sink import SinkGeometry, append_sink_to_swc
from toric_spines_sim.paths import get_pointset_path, get_swc_path

## 1. Build from explicit parameters

The trunk is a straight tapered cable. Spines attach to trunk nodes (never the
proximal neck), each with a thin neck node and a larger head node. Active zones
sit at the head node XYZ — one synapse per spine.

In [2]:
params = SpinyDendriteParams(
    length=10.0,
    trunk_neck_radius=0.5,
    trunk_tip_radius=0.1,
    n_spines=10,
    spine_length=1.0,
    spine_neck_radius=0.1,
    spine_head_radius=0.25,
    spine_neck_length_fraction=0.5,
    max_spines_per_node=2,
    distribution="even",
    azimuth0=0.0,
    axis="z",
)

morph = build_spiny_dendrite(params)

print(f"trunk nodes: {len(morph.trunk_node_ids)}")
print(f"spines / AZ: {morph.n_spines}")
print(f"spines per attach node: {morph.spines_per_attach_node}")
print(f"surface area: {morph.surface_area():.3f} µm²")
print(f"volume:       {morph.volume():.3f} µm³")
print(f"neck point:   {morph.neck_point}")

trunk nodes: 6
spines / AZ: 10
spines per attach node: [2, 2, 2, 2, 2]
surface area: 30.784 µm²
volume:       4.366 µm³
neck point:   (0.0, 0.0, 0.0)


## 2. Write subsystem files

Same convention as toric-spine prep: morphology SWC, AZ pointset, and a single
neck point for later sink attachment.

In [3]:
swc_path = get_swc_path("spiny_dendrite.swc", units="microns")
az_path = get_pointset_path("spiny_dendrite_AZ.txt", units="microns")
neck_path = get_pointset_path("spiny_dendrite_neckpoint.txt", units="microns")

write_subsystem(morph, swc_path, az_path, neck_path)
print(f"wrote {swc_path}")
print(f"wrote {az_path}")
print(f"wrote {neck_path}")

INFO:toric_spines_sim.geometry.dendrite:Wrote subsystem: swc=/home/jordan/repos/toric_spines_sim/data/swc/microns/spiny_dendrite.swc az=/home/jordan/repos/toric_spines_sim/data/pointsets/microns/spiny_dendrite_AZ.txt neck=/home/jordan/repos/toric_spines_sim/data/pointsets/microns/spiny_dendrite_neckpoint.txt (10 synapses)


wrote /home/jordan/repos/toric_spines_sim/data/swc/microns/spiny_dendrite.swc
wrote /home/jordan/repos/toric_spines_sim/data/pointsets/microns/spiny_dendrite_AZ.txt
wrote /home/jordan/repos/toric_spines_sim/data/pointsets/microns/spiny_dendrite_neckpoint.txt


## 3. Visualize morphology + AZ + neck

In [4]:
swc_model = SWCModel.from_swc_file(str(swc_path))
az_points = PointSet.from_txt_file(str(az_path))
neck_points = PointSet.from_txt_file(str(neck_path))

fig = plot_model(
    swc_model=swc_model,
    point_set=az_points,
    point_size=0.15,
    point_color="crimson",
)
fig.show()

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=26 reconnections=0 header=2
INFO:swctools.model:SWCModel.from_parse_result records=26 reconnections=0 header=2
INFO:swctools.model:SWCModel.from_swc_file built nodes=26 edges=25 strict=True validate_reconnections=True
INFO:swctools.geometry:PointSet.from_txt_file path=/home/jordan/repos/toric_spines_sim/data/pointsets/microns/spiny_dendrite_AZ.txt n=10
INFO:swctools.geometry:batch_spheres count=10 stacks=6 slices=12 verts=620 faces=1200
INFO:swctools.geometry:PointSet.from_points n=10 base_radius=1.0 stacks=6 slices=12
INFO:swctools.geometry:PointSet.from_txt_file path=/home/jordan/repos/toric_spines_sim/data/pointsets/microns/spiny_dendrite_neckpoint.txt n=1
INFO:swctools.geometry:batch_spheres count=1 stacks=6 slices=12 verts=62 faces=120
INFO:swctools.geometry:PointSet.from_points n=1 base_radius=1.0 stacks=6 slices=12
INFO:swctools.geometry:batch_frusta c

In [5]:
fig_neck = plot_model(
    swc_model=swc_model,
    point_set=neck_points,
    point_size=0.3,
    point_color="steelblue",
)
fig_neck.show()

INFO:swctools.geometry:batch_frusta count=25 sides=16 end_caps=False verts=800 faces=800
INFO:swctools.geometry:FrustaSet.from_swc_model edges=25 sides=16 end_caps=False
INFO:swctools.geometry:batch_spheres count=1 stacks=6 slices=12 verts=62 faces=120
INFO:swctools.geometry:PointSet.scaled radius_scale=0.3
INFO:swctools.viz:plot_model slider=False frusta=25 show_frusta=True show_centroid=True


## 4. Optional: append a sink at the neck

Downstream of subsystem generation, reuse `append_sink_to_swc` exactly as for
toric spines / the cylinder example.

In [6]:
sink_radius_um = 5.0
swc_with_sink_path = get_swc_path(
    f"spiny_dendrite_wsink_r{sink_radius_um:g}um.swc", units="microns"
)

geom = SinkGeometry(
    radius=sink_radius_um,
    length=2 * sink_radius_um,
    n_cylinders=5,
    axis="-z",
    connector_length=1.0,
)
append_sink_to_swc(
    swc_in=swc_path,
    swc_out=swc_with_sink_path,
    neck_coords=neck_path,
    geom=geom,
    tag=5,
)
print(f"wrote {swc_with_sink_path}")

INFO:swctools.io:parse_swc start strict=True validate_reconnections=False float_tol=1e-09
INFO:swctools.io:parse_swc done records=26 reconnections=0 header=2
INFO:swctools.model:SWCModel.from_parse_result records=26 reconnections=0 header=2
INFO:swctools.model:SWCModel.from_swc_file built nodes=26 edges=25 strict=True validate_reconnections=False


wrote /home/jordan/repos/toric_spines_sim/data/swc/microns/spiny_dendrite_wsink_r5um.swc


In [7]:
swc_with_sink = SWCModel.from_swc_file(str(swc_with_sink_path))
fig_sink = plot_model(
    swc_model=swc_with_sink,
    point_set=az_points,
    point_size=0.15,
    point_color="crimson",
)
fig_sink.show()

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=32 reconnections=0 header=3
INFO:swctools.model:SWCModel.from_parse_result records=32 reconnections=0 header=3
INFO:swctools.model:SWCModel.from_swc_file built nodes=32 edges=31 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=31 sides=16 end_caps=False verts=992 faces=992
INFO:swctools.geometry:FrustaSet.from_swc_model edges=31 sides=16 end_caps=False
INFO:swctools.geometry:batch_spheres count=10 stacks=6 slices=12 verts=620 faces=1200
INFO:swctools.geometry:PointSet.scaled radius_scale=0.15
INFO:swctools.viz:plot_model slider=False frusta=31 show_frusta=True show_centroid=True


## 5. Layout variant: one spine per trunk node

With `max_spines_per_node=1`, the trunk is subdivided so each attach node gets
a single spine (still skipping the proximal neck).

In [8]:
params_single = SpinyDendriteParams(
    length=20.0,
    trunk_neck_radius=0.5,
    trunk_tip_radius=0.35,
    n_spines=8,
    spine_length=1.2,
    spine_neck_radius=0.08,
    spine_head_radius=0.22,
    max_spines_per_node=1,
    distribution="even",
    azimuth0=0.4,
    axis="z",
)
morph_single = build_spiny_dendrite(params_single)
print(f"trunk nodes: {len(morph_single.trunk_node_ids)}")
print(f"spines per attach node: {morph_single.spines_per_attach_node}")

swc_single = get_swc_path("spiny_dendrite_one_per_node.swc", units="microns")
az_single = get_pointset_path("spiny_dendrite_one_per_node_AZ.txt", units="microns")
neck_single = get_pointset_path(
    "spiny_dendrite_one_per_node_neckpoint.txt", units="microns"
)
write_subsystem(morph_single, swc_single, az_single, neck_single)

fig_single = plot_model(
    swc_model=SWCModel.from_swc_file(str(swc_single)),
    point_set=PointSet.from_txt_file(str(az_single)),
    point_size=0.15,
    point_color="darkorange",
)
fig_single.show()

INFO:toric_spines_sim.geometry.dendrite:Wrote subsystem: swc=/home/jordan/repos/toric_spines_sim/data/swc/microns/spiny_dendrite_one_per_node.swc az=/home/jordan/repos/toric_spines_sim/data/pointsets/microns/spiny_dendrite_one_per_node_AZ.txt neck=/home/jordan/repos/toric_spines_sim/data/pointsets/microns/spiny_dendrite_one_per_node_neckpoint.txt (8 synapses)
INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=25 reconnections=0 header=2
INFO:swctools.model:SWCModel.from_parse_result records=25 reconnections=0 header=2
INFO:swctools.model:SWCModel.from_swc_file built nodes=25 edges=24 strict=True validate_reconnections=True
INFO:swctools.geometry:PointSet.from_txt_file path=/home/jordan/repos/toric_spines_sim/data/pointsets/microns/spiny_dendrite_one_per_node_AZ.txt n=8
INFO:swctools.geometry:batch_spheres count=8 stacks=6 slices=12 verts=496 faces=960
INFO:swctools.geometry:PointSet.from_points n=8 base_radiu

trunk nodes: 9
spines per attach node: [1, 1, 1, 1, 1, 1, 1, 1]
